# IMDB Dataset playground

## Setup

In [62]:
import tensorflow as tf
import numpy as np

In [63]:
np.random.seed(42)
tf.random.set_seed(42)

## Data load

In [64]:
from pathlib import Path

root = "https://ai.stanford.edu/~amaas/data/sentiment/"
filename = "aclImdb_v1.tar.gz"
filepath = tf.keras.utils.get_file(filename, root + filename, extract=True,
                                   cache_dir=".")
if "_extracted" in filepath:
    path = Path(filepath) / "aclImdb"
else:
    path = Path(filepath).with_name("aclImdb")

In [65]:
sub_paths = path.iterdir()
for sub_path in sub_paths:
  print(sub_path)

datasets/aclImdb_v1_extracted/aclImdb/train
datasets/aclImdb_v1_extracted/aclImdb/README
datasets/aclImdb_v1_extracted/aclImdb/imdb.vocab
datasets/aclImdb_v1_extracted/aclImdb/test
datasets/aclImdb_v1_extracted/aclImdb/imdbEr.txt


In [36]:
sub_paths = path / "train"
sub_paths = sub_paths.iterdir()
for sub_path in sub_paths:
  print(sub_path)
print()
sub_paths = path / "test"
sub_paths = sub_paths.iterdir()
for sub_path in sub_paths:
  print(sub_path)

datasets/aclImdb_v1_extracted/aclImdb/train/urls_neg.txt
datasets/aclImdb_v1_extracted/aclImdb/train/unsupBow.feat
datasets/aclImdb_v1_extracted/aclImdb/train/labeledBow.feat
datasets/aclImdb_v1_extracted/aclImdb/train/pos
datasets/aclImdb_v1_extracted/aclImdb/train/urls_pos.txt
datasets/aclImdb_v1_extracted/aclImdb/train/unsup
datasets/aclImdb_v1_extracted/aclImdb/train/neg
datasets/aclImdb_v1_extracted/aclImdb/train/urls_unsup.txt

datasets/aclImdb_v1_extracted/aclImdb/test/urls_neg.txt
datasets/aclImdb_v1_extracted/aclImdb/test/labeledBow.feat
datasets/aclImdb_v1_extracted/aclImdb/test/pos
datasets/aclImdb_v1_extracted/aclImdb/test/urls_pos.txt
datasets/aclImdb_v1_extracted/aclImdb/test/neg


In [66]:
train_pos_dr = path / "train" / "pos"
train_neg_dr = path / "train" / "neg"
test_pos_dr = path / "test" / "pos"
test_neg_dr = path / "test"/ "neg"

In [67]:
from pathlib import Path
import numpy as np

def list_txt(dirpath: Path) -> list[Path]:
    return list(dirpath.glob("*.txt"))

# list files as Path objects
train_pos_files = list_txt(train_pos_dr)
train_neg_files = list_txt(train_neg_dr)
test_pos_files  = list_txt(test_pos_dr)
test_neg_files  = list_txt(test_neg_dr)


Split the test set into a validation set (15,000) and a test set (10,000)

In [68]:
# split test set into a validation set (15,000) and a test set (10,000)
rng = np.random.default_rng(42)
rng.shuffle(test_pos_files)
rng.shuffle(test_neg_files)

valid_pos_files = test_pos_files[:5000]
test_pos_files  = test_pos_files[5000:]

valid_neg_files = test_neg_files[:5000]
test_neg_files  = test_neg_files[5000:]


Since the dataset fits in memory, just load all the data using tf.data.Dataset.from_tensor_slices():

In [69]:
from numpy.random import shuffle
from tensorflow.data import Dataset
from typing import Sequence

def make_text_ds(filepaths: Sequence[Path]) -> Dataset:
  paths_as_str = [str(p) for p in filepaths]
  ds = Dataset.from_tensor_slices(paths_as_str)
  ds = ds.map(tf.io.read_file, num_parallel_calls = tf.data.AUTOTUNE)
  return ds

def make_labeled_ds(
    pos_files: Sequence[str],
    neg_files: Sequence[str],
    shuffle=True,
    cache=False):

  pos_ds = make_text_ds(pos_files).map(lambda x: (x, 1))
  neg_ds = make_text_ds(neg_files).map(lambda x: (x, 0))
  ds = pos_ds.concatenate(neg_ds)

  if shuffle:
    ds = ds.shuffle(len(ds))

  if cache:
    ds = ds.cache()

  return ds

train_ds = make_labeled_ds(train_pos_files, train_neg_files, shuffle=True)
valid_ds = make_labeled_ds(valid_pos_files, valid_neg_files, shuffle=False)
test_ds = make_labeled_ds(test_pos_files, test_neg_files, shuffle=False)

In [44]:
for X, y in train_ds.take(3):
  print(X)
  print(y)
  print()

tf.Tensor(b"A man discovers that his parents were part of a nuclear experiment in the 50's and that he now has the power to... burst into flames! <br /><br />I was really geared up for this film, what with being directed by the great Toby Hooper and staring wild card Brad Dourif. Unfortunately it didn't rise above the average individual-with-violent-powers movie. Spontaneous Combustion has an interesting premise behind it, unfortunately it never seems to live up to its potential and prolongs its plot too much. The special effects aren't bad though and help to carry the movie to the finale.<br /><br />The cast isn't bad, Dourif does steal the show.<br /><br />All around, no classic but it's not the worst of its kind either.<br /><br />** out of ****", shape=(), dtype=string)
tf.Tensor(0, shape=(), dtype=int32)

tf.Tensor(b'Foley\'s noir quality in this saturated and intense pulp film is seemingly "perfectly" fit together. Shot by shot, edit by edit, the film unfolds itself around a dist

In [70]:
batch_size = 32
def prefetch(ds, batch_size = 32):
  return ds.batch(batch_size).prefetch(1)

train_set = prefetch(train_ds, batch_size)
valid_set = prefetch(valid_ds, batch_size)
test_set = prefetch(test_ds, batch_size)

## Binary classification

Binary classification model, using a TextVectorization layer to preprocess each review.

Total model contains preprocessing model and binary classification model

In [72]:
# Preprocessing model using TextVectorization, dense representation, adapt
# standardize?, play with ngrams?

max_tokens = 1000
text_vector_layer = tf.keras.layers.TextVectorization(max_tokens=max_tokens,
                                                      output_mode="tf-idf")
# adapt
train_X = train_set.map(lambda X, y: X)
text_vector_layer.adapt(train_X)

In [75]:
text_vector_layer.get_vocabulary()[:10]

['[UNK]',
 np.str_('the'),
 np.str_('and'),
 np.str_('a'),
 np.str_('of'),
 np.str_('to'),
 np.str_('is'),
 np.str_('in'),
 np.str_('it'),
 np.str_('i')]

In [87]:
# binary classification model

model = tf.keras.models.Sequential([
    # Sequential can work without Input layer, but better to add it:
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    text_vector_layer,
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [88]:
# TODO add learning rate
model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=["accuracy"])

In [89]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_4            │ (None, 1000)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 100)            │       100,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,201 (391.41 KB)

 Trainable params: 100,201 (391.41 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
history = model.fit(train_set, epochs=5, validation_data=valid_set)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.7885 - loss: 0.4676 - val_accuracy: 0.8536 - val_loss: 0.3504
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.8579 - loss: 0.3504 - val_accuracy: 0.8374 - val_loss: 0.3961
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.8731 - loss: 0.3106 - val_accuracy: 0.8535 - val_loss: 0.3483
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8993 - loss: 0.2520 - val_accuracy: 0.8472 - val_loss: 0.3699
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.9288 - loss: 0.1920 - val_accuracy: 0.8465 - val_loss: 0.4021


## Add an Embedding layer

Embedding excepts integer, so text_vector_layer needs to be adapted with "int" output mode rather than tf-idf

In case of summing embeddings: longer reviews have larger vector norms =>
Do a variance-stabilizing / length-normalization trick:

 an Embedding layer and compute the mean embedding for each review, multiplied by the square root of the number of words. This rescaled mean embedding can then be passed to the rest of the model.

In [99]:
def compute_mean_embedding(inputs):
    not_pad = tf.math.count_nonzero(inputs, axis=-1)
    n_words = tf.math.count_nonzero(not_pad, axis=-1, keepdims=True)
    sqrt_n_words = tf.math.sqrt(tf.cast(n_words, tf.float32))
    return tf.reduce_sum(inputs, axis=1) / sqrt_n_words

another_example = tf.constant([[[1., 2., 3.], [4., 5., 0.], [0., 0., 0.]],
                               [[6., 0., 0.], [0., 0., 0.], [0., 0., 0.]]])
compute_mean_embedding(another_example)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[3.535534 , 4.9497476, 2.1213205],
       [6.       , 0.       , 0.       ]], dtype=float32)>

In [101]:
max_tokens = 1000
text_to_int_vector_layer = tf.keras.layers.TextVectorization(max_tokens=max_tokens,
                                                      output_mode="int")
# adapt
text_to_int_vector_layer.adapt(train_X)

In [102]:
model = tf.keras.models.Sequential([
    # Sequential can work without Input layer, but better to add it:
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    text_to_int_vector_layer,
    tf.keras.layers.Embedding(input_dim=max_tokens, output_dim=max_tokens, mask_zero=True),
    tf.keras.layers.Lambda(compute_mean_embedding),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'lambda' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [ ]:
model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=["accuracy"])

history = model.fit(train_set, epochs=5, validation_data=valid_set)

Epoch 1/5
157/782 ━━━━━━━━━━━━━━━━━━━━ 4:04 391ms/step - accuracy: 0.4977 - loss: 0.8083